# 0.1 Clean Ru-AHO and Build Ru-AHK Domains

This notebook is the single data-preparation entry point for the v2 Ru-centered transfer-learning study.

It performs two independent workflows:

1. **Ru-AHO target-domain preparation**
   - Read `data/raw_data/aho_dataset.csv`.
   - Standardize structures and retain the predefined terminal, di-substituted Ru-AHO reactions.
   - Verify the reactant-to-product hydrogenation relationship and remove duplicates.
   - Write `data/raw_data/ru_aho_2020_2025.csv` for downstream target-domain modeling.

2. **Ru-AHK source-domain preparation**
   - Read the Ru-only literature table `data/raw_data/ahk_dataset_Ru.csv`.
   - Standardize the reaction fields and calculate ΔΔG‡ from ee.
   - Retain the existing `Mechanism` labels provided in the raw dataset.
   - Remove records without a mechanism label.
   - Write the complete labeled table to `data/all_ahk_Ru/processed_ahk_dataset_Ru.csv`.
   - Split it into `data/transfer/ru_ahk_inner.csv` and `data/transfer/ru_ahk_outer.csv`.


In [1]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or script location."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists():
            return candidate

    # Fallback when the notebook is opened before data/ has been created.
    if start.name == "0_data_preparation" and start.parent.name == "notebooks":
        return start.parents[1]
    return start


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_ROOT = DATA_ROOT / "raw_data"
ALL_RU_AHK_ROOT = DATA_ROOT / "all_ahk_Ru"
TRANSFER_DATA_ROOT = DATA_ROOT / "transfer"
RESULTS_ROOT = PROJECT_ROOT / "results"

for directory in [RAW_DATA_ROOT, ALL_RU_AHK_ROOT, TRANSFER_DATA_ROOT, RESULTS_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

AHO_RAW_PATH = RAW_DATA_ROOT / "aho_dataset.csv"
RU_AHO_TARGET_PATH = TRANSFER_DATA_ROOT / "ru_aho_inner.csv"
RU_AHK_RAW_PATH = RAW_DATA_ROOT / "ahk_dataset_Ru.csv"
PROCESSED_RU_AHK_PATH = ALL_RU_AHK_ROOT / "processed_ahk_dataset_Ru.csv"
RU_AHK_INNER_PATH = TRANSFER_DATA_ROOT / "ru_ahk_inner.csv"
RU_AHK_OUTER_PATH = TRANSFER_DATA_ROOT / "ru_ahk_outer.csv"


print(f"Project root: {PROJECT_ROOT}")
print(f"AHO input: {AHO_RAW_PATH}")
print(f"Ru-AHO target output: {RU_AHO_TARGET_PATH}")
print(f"Ru-AHK input: {RU_AHK_RAW_PATH}")
print(f"Processed Ru-AHK output: {PROCESSED_RU_AHK_PATH}")
print(f"Transfer-domain output root: {TRANSFER_DATA_ROOT}")

Project root: /home/syz/ru-mechanism-transfer
AHO input: /home/syz/ru-mechanism-transfer/data/raw_data/aho_dataset.csv
Ru-AHO target output: /home/syz/ru-mechanism-transfer/data/transfer/ru_aho_inner.csv
Ru-AHK input: /home/syz/ru-mechanism-transfer/data/raw_data/ahk_dataset_Ru.csv
Processed Ru-AHK output: /home/syz/ru-mechanism-transfer/data/all_ahk_Ru/processed_ahk_dataset_Ru.csv
Transfer-domain output root: /home/syz/ru-mechanism-transfer/data/transfer


## 1. Shared cleaning and validation utilities

The helpers below standardize missing values and SMILES, verify terminal-alkene hydrogenation, calculate ΔΔG‡, and perform deterministic deduplication.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, RDLogger
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
RDLogger.DisableLog("rdApp.warning")


R_KCAL = 0.0019872

DOMAIN_COLUMNS = [
    "Reactant SMILES",
    "Product SMILES",
    "Metal",
    "Ligand SMILES",
    "Solvent SMILES",
    "Additive SMILES",
    "Pressure/atm",
    "Temperature/C",
    "S/C",
    "ddG",
]

PROCESSED_RU_AHK_COLUMNS = [
    "Reactant SMILES",
    "Product SMILES",
    "Reaction Year",
    "Metal",
    "Ligand SMILES",
    "Solvent SMILES",
    "Additive SMILES",
    "Pressure/atm",
    "Temperature/C",
    "S/C",
    "ddG",
    "Mechanism",
]


def require_file(path: Path) -> None:
    """Fail early when a required repository input is absent."""
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file was not found: {path}\n"
            "Run this notebook from the repository or place the file at the documented path."
        )


def require_columns(df: pd.DataFrame, columns: list[str], table_name: str) -> None:
    """Raise a clear error if required columns are absent."""
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise KeyError(f"{table_name} is missing required columns: {missing}")


def clean_string(value) -> str:
    """Normalize missing string-like values to an empty string."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null"}:
        return ""
    return text


def merge_smiles(row: pd.Series, columns: list[str]) -> str:
    """Concatenate non-empty component SMILES with '.'."""
    values = []
    for column in columns:
        if column in row.index:
            text = clean_string(row[column])
            if text:
                values.append(text)
    return ".".join(values)


def standardize_smiles(smiles):
    """Return canonical isomeric SMILES; invalid or empty entries become None."""
    text = clean_string(smiles)
    if not text:
        return None
    molecule = Chem.MolFromSmiles(text)
    if molecule is None:
        return None
    return Chem.MolToSmiles(molecule, canonical=True, isomericSmiles=True)


def has_terminal_alkene(smiles) -> bool:
    """Detect a terminal alkene motif, CH2=C."""
    text = clean_string(smiles)
    if not text:
        return False
    molecule = Chem.MolFromSmiles(text)
    if molecule is None:
        return False
    pattern = Chem.MolFromSmarts("[CH2]=[C]")
    return bool(pattern is not None and molecule.HasSubstructMatch(pattern))


def reaction_matches_product(
    reactant_smiles, product_smiles, alkene_smarts: str = "[CH2]=[C]"
) -> bool:
    """Check whether reducing one selected C=C bond generates the reported product."""
    reactant_text = clean_string(reactant_smiles)
    product_text = clean_string(product_smiles)
    if not reactant_text or not product_text:
        return False

    reactant = Chem.MolFromSmiles(reactant_text)
    product = Chem.MolFromSmiles(product_text)
    if reactant is None or product is None:
        return False

    product_key = Chem.MolToSmiles(product, canonical=True, isomericSmiles=False)
    alkene_pattern = Chem.MolFromSmarts(alkene_smarts)
    if alkene_pattern is None:
        return False

    for carbon_1_index, carbon_2_index in reactant.GetSubstructMatches(alkene_pattern):
        bond = reactant.GetBondBetweenAtoms(carbon_1_index, carbon_2_index)
        if bond is None or bond.GetBondType() != Chem.rdchem.BondType.DOUBLE:
            continue

        editable = Chem.RWMol(reactant)
        editable.GetBondWithIdx(bond.GetIdx()).SetBondType(Chem.rdchem.BondType.SINGLE)
        try:
            reduced = editable.GetMol()
            Chem.SanitizeMol(reduced)
            reduced_key = Chem.MolToSmiles(reduced, canonical=True, isomericSmiles=False)
            if reduced_key == product_key:
                return True
        except Exception:
            continue
    return False


def deduplicate_keep_max_ddg(df: pd.DataFrame, subset_columns: list[str] | None = None) -> pd.DataFrame:
    """Remove identical reaction-condition records, retaining the largest ΔΔG‡."""
    if df.empty:
        return df.reset_index(drop=True)

    output = df.copy()
    output["ddG"] = pd.to_numeric(output["ddG"], errors="coerce")
    if subset_columns is None:
        subset_columns = [column for column in output.columns if column != "ddG"]

    return (
        output.assign(_ddg_sort=output["ddG"].fillna(-np.inf))
        .sort_values("_ddg_sort", ascending=False)
        .drop(columns="_ddg_sort")
        .drop_duplicates(subset=subset_columns, keep="first")
        .reset_index(drop=True)
    )


def assert_only_ru(series: pd.Series, table_name: str) -> None:
    """Ensure that a nominally Ru-only table contains no other metal labels."""
    metals = sorted({clean_string(value) for value in series if clean_string(value)})
    if metals != ["Ru"]:
        raise ValueError(f"{table_name} must contain only Ru entries; observed labels: {metals}")


def build_domain_table(df: pd.DataFrame) -> pd.DataFrame:
    """Return the common schema consumed by downstream transfer-learning notebooks."""
    require_columns(df, DOMAIN_COLUMNS, "domain table")
    return df.loc[:, DOMAIN_COLUMNS].copy()


def print_filter_step(label: str, before: int, after: int) -> None:
    """Print a compact row-count audit for one filtering step."""
    print(f"{label}: {before} -> {after} (removed {before - after})")

## 2. Prepare the Ru-AHO target-domain file

The AHO workflow follows the established target-domain definition: valid structures, `di-sub` olefins, a terminal alkene, a verified alkene-to-alkane transformation, and a Ru metal center. Deduplication retains the largest reported ΔΔG‡ for otherwise identical records.

In [3]:
require_file(AHO_RAW_PATH)
aho_raw = pd.read_csv(AHO_RAW_PATH)
print("Raw AHO shape:", aho_raw.shape)

require_columns(
    aho_raw,
    [
        "Reactant SMILES",
        "Product SMILES",
        "Metal",
        "Ligand SMILES",
        "Solvent SMILES",
        "Additive SMILES",
        "Pressure/atm",
        "Temperature/C",
        "S/C",
        "ddG",
        "Olefin Type",
    ],
    "raw AHO table",
)

aho_clean = pd.DataFrame()
aho_clean["Reactant SMILES"] = aho_raw["Reactant SMILES"].apply(standardize_smiles)
aho_clean["Product SMILES"] = aho_raw["Product SMILES"].apply(standardize_smiles)
aho_clean["Metal"] = aho_raw["Metal"].map(clean_string)
aho_clean["Ligand SMILES"] = aho_raw["Ligand SMILES"].map(clean_string)
aho_clean["Solvent SMILES"] = aho_raw["Solvent SMILES"].map(clean_string)
aho_clean["Additive SMILES"] = aho_raw["Additive SMILES"].map(clean_string)
aho_clean["Pressure/atm"] = pd.to_numeric(aho_raw["Pressure/atm"], errors="coerce")
aho_clean["Temperature/C"] = pd.to_numeric(aho_raw["Temperature/C"], errors="coerce")
aho_clean["S/C"] = pd.to_numeric(aho_raw["S/C"], errors="coerce")
aho_clean["ddG"] = pd.to_numeric(aho_raw["ddG"], errors="coerce")
aho_clean["Olefin Type"] = aho_raw["Olefin Type"].map(clean_string)
for column in ["Catalyst SMILES(RDKit)", "DOI"]:
    if column in aho_raw.columns:
        aho_clean[column] = aho_raw[column].map(clean_string)
for column in ["Yield", "ee"]:
    if column in aho_raw.columns:
        aho_clean[column] = pd.to_numeric(aho_raw[column], errors="coerce")

before = len(aho_clean)
valid_structure_mask = aho_clean["Reactant SMILES"].notna() & aho_clean["Product SMILES"].notna()
aho_clean = aho_clean.loc[valid_structure_mask].reset_index(drop=True)
print_filter_step("Valid reactant/product structures", before, len(aho_clean))
before = len(aho_clean)
di_sub_mask = aho_clean["Olefin Type"].str.lower().eq("di-sub")
aho_clean = aho_clean.loc[di_sub_mask].reset_index(drop=True)
print_filter_step("Olefin Type = di-sub", before, len(aho_clean))

before = len(aho_clean)
terminal_mask = aho_clean["Reactant SMILES"].apply(has_terminal_alkene)
aho_clean = aho_clean.loc[terminal_mask].reset_index(drop=True)
print_filter_step("Terminal alkene present", before, len(aho_clean))

before = len(aho_clean)
reaction_match_mask = aho_clean.apply(
    lambda row: reaction_matches_product(row["Reactant SMILES"], row["Product SMILES"]),
    axis=1,
)
aho_clean = aho_clean.loc[reaction_match_mask].reset_index(drop=True)
print_filter_step("Verified reactant-to-product hydrogenation", before, len(aho_clean))

aho_clean["metal_std"] = (
    aho_clean["Metal"].astype(str).str.extract(r"(Rh|Ir|Ru|Co)", expand=False)
)
before = len(aho_clean)
ru_aho_target = aho_clean.loc[aho_clean["metal_std"].eq("Ru")].copy()
print_filter_step("Ru metal center", before, len(ru_aho_target))

ru_aho_target = build_domain_table(ru_aho_target)
before = len(ru_aho_target)
ru_aho_target = deduplicate_keep_max_ddg(ru_aho_target)
print_filter_step("AHO reaction-condition deduplication", before, len(ru_aho_target))

if ru_aho_target.empty:
    raise ValueError("The Ru-AHO target-domain table is empty after filtering.")
if ru_aho_target["ddG"].isna().any():
    raise ValueError("The Ru-AHO target-domain table contains missing ddG values.")

ru_aho_target.to_csv(RU_AHO_TARGET_PATH, index=False)
print(f"Saved Ru-AHO target domain ({len(ru_aho_target)} rows) to:\n{RU_AHO_TARGET_PATH}")
display(ru_aho_target.head())

Raw AHO shape: (12617, 23)
Valid reactant/product structures: 12617 -> 12617 (removed 0)
Olefin Type = di-sub: 12617 -> 3312 (removed 9305)
Terminal alkene present: 3312 -> 3310 (removed 2)
Verified reactant-to-product hydrogenation: 3310 -> 3159 (removed 151)
Ru metal center: 3159 -> 109 (removed 3050)
AHO reaction-condition deduplication: 109 -> 98 (removed 11)
Saved Ru-AHO target domain (98 rows) to:
/home/syz/ru-mechanism-transfer/data/transfer/ru_aho_inner.csv


,Reactant SMILES,Product SMILES,Metal,Ligand SMILES,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C,ddG
0,C=C(C(=O)O)c1cccc2ccccc12,C[C@H](C(=O)O)c1cccc2ccccc12,Ru,CC(C)(C)[C@H]1COC(C2=C(P(c3ccccc3)c3ccccc3)C([...,CO,O=C([O-])O.[Na+],5.0,25.0,100.0,4.500500
1,C=C(C(=O)O)c1ccccc1Cl,C[C@H](C(=O)O)c1ccccc1Cl,Ru,CC(C)(C)[C@H]1COC(C2=C(P(c3ccccc3)c3ccccc3)C([...,CO,O=C([O-])O.[Na+],5.0,25.0,100.0,4.500500
2,C=C(C(=O)O)c1ccccc1F,C[C@H](C(=O)O)c1ccccc1F,Ru,CC(C)(C)[C@H]1COC(C2=C(P(c3ccccc3)c3ccccc3)C([...,CO,O=C([O-])O.[Na+],5.0,25.0,100.0,3.849375
3,C=C(CC)C(=O)O,CC[C@H](C)C(=O)O,Ru,CC(C)(C)[C@H]1COC(C2=C(P(c3ccccc3)c3ccccc3)C([...,CO,O=C([O-])O.[Na+],5.0,25.0,100.0,3.678731
4,C=C(C(=O)O)c1ccc(F)cc1,C[C@H](C(=O)O)c1ccc(F)cc1,Ru,CC(C)(C)[C@H]1COC(C2=C(P(c3ccccc3)c3ccccc3)C([...,CO,O=C([O-])O.[Na+],5.0,25.0,100.0,3.134376


## 3. Clean the Ru-AHK dataset

The input file is already restricted to Ru catalysis and already contains the literature-based `Mechanism` annotation. This notebook does not infer mechanism from ligand structure; it only cleans the reaction records and preserves the existing mechanism labels.

Records without a mechanism label are removed before deduplication.

In [4]:
require_file(RU_AHK_RAW_PATH)
ru_ahk_raw = pd.read_csv(RU_AHK_RAW_PATH)
print("Raw Ru-AHK shape:", ru_ahk_raw.shape)

require_columns(
    ru_ahk_raw,
    [
        "Reactant SMILES 1",
        "Product SMILES 1",
        "Reaction Year",
        "Catalyst Central Metal 1",
        "Catalyst Ammount (eq) 1",
        "Catalyst Ligand SMILES 1",
        "Catalyst Ligand SMILES 2",
        "Step Pressure (atm) 1",
        "Step Temperature (C) 1",
        "Product ee (%) 1",
        "Mechanism",
    ],
    "raw Ru-AHK table",
)
assert_only_ru(ru_ahk_raw["Catalyst Central Metal 1"], "raw Ru-AHK table")

ru_ahk_enriched = ru_ahk_raw.copy()
ru_ahk_enriched["Mechanism"] = ru_ahk_enriched["Mechanism"].map(clean_string)
ru_ahk_enriched["Solvent SMILES"] = ru_ahk_enriched.apply(
    lambda row: merge_smiles(row, ["Solvent SMILES 1", "Solvent SMILES 2", "Solvent SMILES 3"]),
    axis=1,
)
ru_ahk_enriched["Ligand SMILES"] = ru_ahk_enriched.apply(
    lambda row: merge_smiles(row, ["Catalyst Ligand SMILES 1", "Catalyst Ligand SMILES 2"]),
    axis=1,
)
ru_ahk_enriched["Additive SMILES"] = ru_ahk_enriched.apply(
    lambda row: merge_smiles(row, ["Additive SMILES 1", "Additive SMILES 2", "Additive SMILES 3"]),
    axis=1,
)

catalyst_amount = pd.to_numeric(
    ru_ahk_enriched["Catalyst Ammount (eq) 1"], errors="coerce"
)
ru_ahk_enriched["S/C"] = 1 / catalyst_amount.replace(0, np.nan)

reaction_temperature_k = (
    pd.to_numeric(ru_ahk_enriched["Step Temperature (C) 1"], errors="coerce") + 273.15
)
ee = pd.to_numeric(ru_ahk_enriched["Product ee (%) 1"], errors="coerce").clip(
    lower=0, upper=99.9
)
enantiomeric_ratio = (1 + ee / 100) / (1 - ee / 100)
ru_ahk_enriched["ddG"] = R_KCAL * reaction_temperature_k * np.log(enantiomeric_ratio)

print("Raw mechanism-label counts:")
print(
    ru_ahk_enriched["Mechanism"]
    .replace("", "unlabeled")
    .value_counts(dropna=False)
    .rename_axis("mechanism")
    .to_frame("rows")
)

unlabeled_count = int(ru_ahk_enriched["Mechanism"].eq("").sum())
print(f"Unlabeled raw Ru-AHK rows to remove: {unlabeled_count}")

Raw Ru-AHK shape: (2173, 38)
Raw mechanism-label counts:
           rows
mechanism      
outer      1632
inner       541
Unlabeled raw Ru-AHK rows to remove: 0


## 4. Export the complete labeled Ru-AHK table

Unlabeled records are removed first. The remaining records are converted to the common processed schema and deduplicated across reaction structures and experimental conditions while retaining the largest ΔΔG‡.

In [5]:
ru_ahk_labeled = ru_ahk_enriched.loc[ru_ahk_enriched["Mechanism"].ne("")].copy()

processed_ru_ahk = pd.DataFrame()
processed_ru_ahk["Reactant SMILES"] = ru_ahk_labeled["Reactant SMILES 1"].map(clean_string)
processed_ru_ahk["Product SMILES"] = ru_ahk_labeled["Product SMILES 1"].map(clean_string)
processed_ru_ahk["Reaction Year"] = pd.to_numeric(
    ru_ahk_labeled["Reaction Year"], errors="coerce"
).astype("Int64")
processed_ru_ahk["Metal"] = ru_ahk_labeled["Catalyst Central Metal 1"].map(clean_string)
processed_ru_ahk["Ligand SMILES"] = ru_ahk_labeled["Ligand SMILES"].map(clean_string)
processed_ru_ahk["Solvent SMILES"] = ru_ahk_labeled["Solvent SMILES"].map(clean_string)
processed_ru_ahk["Additive SMILES"] = ru_ahk_labeled["Additive SMILES"].map(clean_string)
processed_ru_ahk["Pressure/atm"] = pd.to_numeric(
    ru_ahk_labeled["Step Pressure (atm) 1"], errors="coerce"
)
processed_ru_ahk["Temperature/C"] = pd.to_numeric(
    ru_ahk_labeled["Step Temperature (C) 1"], errors="coerce"
)
processed_ru_ahk["S/C"] = pd.to_numeric(ru_ahk_labeled["S/C"], errors="coerce")
processed_ru_ahk["ddG"] = pd.to_numeric(ru_ahk_labeled["ddG"], errors="coerce")
processed_ru_ahk["Mechanism"] = ru_ahk_labeled["Mechanism"].astype(str)
processed_ru_ahk = processed_ru_ahk.loc[:, PROCESSED_RU_AHK_COLUMNS]

before = len(processed_ru_ahk)
processed_ru_ahk = deduplicate_keep_max_ddg(processed_ru_ahk)
print_filter_step("Ru-AHK reaction-condition deduplication", before, len(processed_ru_ahk))

assert_only_ru(processed_ru_ahk["Metal"], "processed Ru-AHK table")
if processed_ru_ahk["Mechanism"].isna().any() or processed_ru_ahk["Mechanism"].eq("").any():
    raise AssertionError("Processed Ru-AHK still contains unlabeled records.")
if not set(processed_ru_ahk["Mechanism"].unique()).issubset({"inner", "outer"}):
    raise AssertionError("Unexpected mechanism labels were found.")
if processed_ru_ahk["ddG"].isna().any():
    raise ValueError("Processed Ru-AHK contains missing ddG values.")

processed_ru_ahk.to_csv(PROCESSED_RU_AHK_PATH, index=False)
print(f"Saved processed mechanistically labeled Ru-AHK dataset to:\n{PROCESSED_RU_AHK_PATH}")
print("Processed mechanism counts:")
print(processed_ru_ahk["Mechanism"].value_counts())
display(processed_ru_ahk.head())

Ru-AHK reaction-condition deduplication: 2173 -> 2034 (removed 139)
Saved processed mechanistically labeled Ru-AHK dataset to:
/home/syz/ru-mechanism-transfer/data/all_ahk_Ru/processed_ahk_dataset_Ru.csv
Processed mechanism counts:
Mechanism
outer    1539
inner     495
Name: count, dtype: int64


,Reactant SMILES,Product SMILES,Reaction Year,Metal,Ligand SMILES,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C,ddG,Mechanism
0,O=C(COCc1ccccc1)CC(=O)c1ccccc1,O=C(C[C@@H](O)COCc1ccccc1)c1ccccc1,2019,Ru,CC1(C)Oc2c(O1)c(c1c3OC(Oc3ccc1P(c1ccccc1)c1ccc...,C1CCCO1,,20.00,70.0,200.0,5.182773,inner
1,CC(=O)CC(=O)COCc1ccccc1,CC(=O)C[C@@H](O)COCc1ccccc1,2019,Ru,CC1(C)Oc2c(O1)c(c1c3OC(Oc3ccc1P(c1ccccc1)c1ccc...,C1CCCO1,,20.00,90.0,200.0,4.984271,inner
2,CC(=O)c1ccc(Br)cc1,C[C@H](O)c1ccc(Br)cc1,2002,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1cc(C)cc(c1)C)c...,CC(O)C,[O-1]C(C)(C)C.[K+],20.41,26.5,50000.0,4.525770,outer
3,CC(=O)c1ccccc1Br,C[C@H](O)c1ccccc1Br,2002,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1ccccc1)c1ccccc...,CC(O)C,[O-1]C(C)(C)C.[K+],20.41,26.5,4000.0,4.525770,outer
4,CC(=O)c1ccccc1Br,C[C@H](O)c1ccccc1Br,2002,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1cc(C)cc(c1)C)c...,CC(O)C,[O-1]C(C)(C)C.[K+],27.22,26.5,10000.0,4.525770,outer


## 5. Split the labeled Ru-AHK table into transfer source domains

The two transfer files are strict partitions of `processed_ahk_dataset_Ru.csv`. The downstream modeling schema remains unchanged and does not include the mechanism label because domain identity is encoded by the file itself.

In [6]:
ru_ahk_inner = build_domain_table(
    processed_ru_ahk.loc[processed_ru_ahk["Mechanism"].eq("inner")]
).reset_index(drop=True)
ru_ahk_outer = build_domain_table(
    processed_ru_ahk.loc[processed_ru_ahk["Mechanism"].eq("outer")]
).reset_index(drop=True)

ru_ahk_inner.to_csv(RU_AHK_INNER_PATH, index=False)
ru_ahk_outer.to_csv(RU_AHK_OUTER_PATH, index=False)

print(f"Saved Ru-AHK inner source ({len(ru_ahk_inner)} rows) to:\n{RU_AHK_INNER_PATH}")
print(f"Saved Ru-AHK outer source ({len(ru_ahk_outer)} rows) to:\n{RU_AHK_OUTER_PATH}")

display(ru_ahk_inner.head())
display(ru_ahk_outer.head())

Saved Ru-AHK inner source (495 rows) to:
/home/syz/ru-mechanism-transfer/data/transfer/ru_ahk_inner.csv
Saved Ru-AHK outer source (1539 rows) to:
/home/syz/ru-mechanism-transfer/data/transfer/ru_ahk_outer.csv


,Reactant SMILES,Product SMILES,Metal,Ligand SMILES,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C,ddG
0,O=C(COCc1ccccc1)CC(=O)c1ccccc1,O=C(C[C@@H](O)COCc1ccccc1)c1ccccc1,Ru,CC1(C)Oc2c(O1)c(c1c3OC(Oc3ccc1P(c1ccccc1)c1ccc...,C1CCCO1,,20.0,70.0,200.0,5.182773
1,CC(=O)CC(=O)COCc1ccccc1,CC(=O)C[C@@H](O)COCc1ccccc1,Ru,CC1(C)Oc2c(O1)c(c1c3OC(Oc3ccc1P(c1ccccc1)c1ccc...,C1CCCO1,,20.0,90.0,200.0,4.984271
2,CN(CCC(=O)c1ccccc1)Cc1ccccc1,CN(CC[C@@H](O)c1ccccc1)Cc1ccccc1,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,Cc1ccccc1.O,[OH-1].[K+],20.0,25.0,2000.0,4.503114
3,CN(CCC(=O)c1ccc(Cl)cc1)Cc1ccccc1,CN(CC[C@@H](O)c1ccc(Cl)cc1)Cc1ccccc1,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,Cc1ccccc1.O,[OH-1].[K+],20.0,25.0,400.0,4.503114
4,Cc1ccc(C(=O)CCN(C)Cc2ccccc2)cc1,Cc1ccc([C@H](O)CCN(C)Cc2ccccc2)cc1,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,Cc1ccccc1.O,[OH-1].[K+],20.0,25.0,400.0,4.503114


,Reactant SMILES,Product SMILES,Metal,Ligand SMILES,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C,ddG
0,CC(=O)c1ccc(Br)cc1,C[C@H](O)c1ccc(Br)cc1,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1cc(C)cc(c1)C)c...,CC(O)C,[O-1]C(C)(C)C.[K+],20.41,26.5,50000.0,4.525770
1,CC(=O)c1ccccc1Br,C[C@H](O)c1ccccc1Br,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1ccccc1)c1ccccc...,CC(O)C,[O-1]C(C)(C)C.[K+],20.41,26.5,4000.0,4.525770
2,CC(=O)c1ccccc1Br,C[C@H](O)c1ccccc1Br,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1cc(C)cc(c1)C)c...,CC(O)C,[O-1]C(C)(C)C.[K+],27.22,26.5,10000.0,4.525770
3,CC(=O)c1ccccc1Br,C[C@H](O)c1ccccc1Br,Ru,COc1nc(OC)cc(c1c1c(OC)nc(cc1P(c1ccc(cc1)C)c1cc...,CC(O)C,[O-1]C(C)(C)C.[K+],20.41,26.5,4000.0,4.525770
4,COc1ccc(C(C)=O)cc1,COc1ccc([C@H](C)O)cc1,Ru,Cc1cc(C)cc(c1)P(c1ccc2c(c1c1c(ccc3c1cccc3)P(c1...,CC(O)C,[O-1]C(C)(C)C.[K+],48.30,25.0,1000.0,4.503114


## 6. Final integrity checks and output summary

These checks ensure that the two source domains are mutually exclusive, exhaustive with respect to the processed Ru-AHK table, and consistent with the fixed publication dataset counts.

In [7]:
observed_counts = {
    "processed": len(processed_ru_ahk),
    "inner": len(ru_ahk_inner),
    "outer": len(ru_ahk_outer),
}

# A chemistry-condition key is used to confirm that no record occurs in both source files.
source_key_columns = [column for column in DOMAIN_COLUMNS if column != "ddG"]
inner_keys = set(map(tuple, ru_ahk_inner[source_key_columns].fillna("").to_numpy()))
outer_keys = set(map(tuple, ru_ahk_outer[source_key_columns].fillna("").to_numpy()))
if inner_keys & outer_keys:
    raise AssertionError("The Ru-AHK inner and outer source domains are not mutually exclusive.")

summary = pd.DataFrame(
    [
        {
            "dataset": "ru_aho_target",
            "rows": len(ru_aho_target),
            "path": str(RU_AHO_TARGET_PATH),
        },
        {
            "dataset": "processed_ru_ahk",
            "rows": len(processed_ru_ahk),
            "path": str(PROCESSED_RU_AHK_PATH),
        },
        {
            "dataset": "ru_ahk_inner",
            "rows": len(ru_ahk_inner),
            "path": str(RU_AHK_INNER_PATH),
        },
        {
            "dataset": "ru_ahk_outer",
            "rows": len(ru_ahk_outer),
            "path": str(RU_AHK_OUTER_PATH),
        },
    ]
)

display(summary)
print("All data-preparation checks passed.")

,dataset,rows,path
0,ru_aho_target,98,/home/syz/ru-mechanism-transfer/data/transfer/...
1,processed_ru_ahk,2034,/home/syz/ru-mechanism-transfer/data/all_ahk_R...
2,ru_ahk_inner,495,/home/syz/ru-mechanism-transfer/data/transfer/...
3,ru_ahk_outer,1539,/home/syz/ru-mechanism-transfer/data/transfer/...


All data-preparation checks passed.
